# SenseBand - ML model training (M1, M3, M4, M5)

Self-contained: this notebook writes its own code files, downloads the public datasets, trains the models with **subject-wise
cross-validation**, exports **ONNX**, and prints the metrics. Run top to bottom (about 10 minutes, ~1 GB download, Python 3.10+).

| Model | Task | Real data used | Result in our run | Target |
|---|---|---|---|---|
| M1 | activity & posture | UCI HAR + PAMAP2 | macro-F1 ~0.85 | 0.90 (not met) |
| M3 | ECG signal quality | PhysioNet CinC 2017 (+ synthetic unusable) | accuracy ~0.96 | 0.92 (met) |
| M4 | rhythm screening ("possible AF") | PhysioNet CinC 2017 | AF F1 ~0.74 | 0.80 (not met) |
| M5 | expected heart rate + per-user RLS | PAMAP2 | MAE ~10 bpm | 8 bpm (not met) |

**Not in this notebook:** M2 (fall detection). The only data we could reach was *simulated* falls from the SenseBand simulator, so
that model is not validated and is kept off. SenseBand is a monitoring aid, not a medical device: these models do not diagnose.
Numbers will vary slightly between runs/machines. The datasets are third-party: respect their licences/terms.

## 1. Install

In [ ]:
%pip install -q numpy scipy scikit-learn lightgbm onnx onnxruntime onnxmltools skl2onnx pyyaml

## 2. Write the shared feature code and training code
These files are the exact ones used in the SenseBand project (one feature implementation for training and for the band).

In [ ]:
import os
for d in ('senseband_edge/models', 'senseband_features', 'senseband_train'):
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile senseband_edge/__init__.py
# (package marker)


In [ ]:
%%writefile senseband_edge/models/__init__.py
# (package marker)


In [ ]:
%%writefile senseband_edge/models/personal.py
"""Per-user correction of the population expected-HR model (docs/04, M5): recursive least squares.

corrected = c0 + c1 * z   with z = (population_prediction - 100) / 50, so the starting point (c0=100, c1=50)
is the identity. RLS with a forgetting factor learns THIS person's resting level and slope during the first
48 h (the "learning period") and keeps adapting slowly. Pure Python: runs on a Pi Zero.
"""
from __future__ import annotations

from typing import Any


class PersonalHr:
    SLOPE_RANGE = (25.0, 80.0)        # c1: 0.5x .. 1.6x of the population slope
    MAX_OFFSET = 40.0                 # c0 may move at most 40 bpm from 100

    def __init__(self, forgetting: float = 0.999, var_offset: float = 400.0, var_slope: float = 100.0):
        self.lam = forgetting
        self.c = [100.0, 50.0]
        self.p = [[var_offset, 0.0], [0.0, var_slope]]
        self.n = 0

    @staticmethod
    def _z(pred: float) -> float:
        return (pred - 100.0) / 50.0

    def correct(self, pred: float) -> float:
        return self.c[0] + self.c[1] * self._z(pred)

    def update(self, pred: float, actual: float) -> None:
        """Feed one (population prediction, measured HR) pair from a calm, good-quality window."""
        x = [1.0, self._z(pred)]
        px = [self.p[0][0] * x[0] + self.p[0][1] * x[1], self.p[1][0] * x[0] + self.p[1][1] * x[1]]
        denom = self.lam + x[0] * px[0] + x[1] * px[1]
        k = [px[0] / denom, px[1] / denom]
        err = actual - (self.c[0] * x[0] + self.c[1] * x[1])
        self.c[0] += k[0] * err
        self.c[1] += k[1] * err
        xp = [x[0] * self.p[0][0] + x[1] * self.p[1][0], x[0] * self.p[0][1] + x[1] * self.p[1][1]]
        for i in range(2):
            for j in range(2):
                self.p[i][j] = (self.p[i][j] - k[i] * xp[j]) / self.lam
        self.c[1] = min(max(self.c[1], self.SLOPE_RANGE[0]), self.SLOPE_RANGE[1])
        self.c[0] = min(max(self.c[0], 100.0 - self.MAX_OFFSET), 100.0 + self.MAX_OFFSET)
        self.n += 1

    def state(self) -> dict[str, Any]:
        return {"c": list(self.c), "p": [list(r) for r in self.p], "n": self.n, "lam": self.lam}

    @classmethod
    def from_state(cls, s: dict[str, Any]) -> PersonalHr:
        o = cls(float(s.get("lam", 0.999)))
        o.c, o.p, o.n = [float(v) for v in s["c"]], [[float(v) for v in r] for r in s["p"]], int(s["n"])
        return o


In [ ]:
%%writefile senseband_features/__init__.py
"""Shared feature extraction used by BOTH training (ml/train) and the edge service (edge/senseband_edge).

One implementation = no train/serve skew (CLAUDE.md). Pure numpy/scipy, no sklearn, so the Pi can import it.
Feature ORDER is part of the contract: models are trained against `*_FEATURES` and the edge feeds them in the
same order. Bump FEATURE_VERSION on any change (the model metrics file records it; the edge refuses a mismatch).
"""
from .common import FEATURE_VERSION
from .ecg import ECG_QUALITY_FEATURES, RR_FEATURES, ecg_quality_features, r_peaks, rr_features, rr_intervals
from .fall import FALL_FEATURES, fall_window_features
from .hr import ACTIVITY_CODES, EXPECTED_HR_FEATURES, expected_hr_features
from .imu import IMU_FEATURES, imu_window_features

__all__ = [
    "ACTIVITY_CODES", "ECG_QUALITY_FEATURES", "EXPECTED_HR_FEATURES", "FALL_FEATURES", "FEATURE_VERSION",
    "IMU_FEATURES", "RR_FEATURES", "ecg_quality_features", "expected_hr_features", "fall_window_features",
    "imu_window_features", "r_peaks", "rr_features", "rr_intervals",
]


In [ ]:
%%writefile senseband_features/common.py
"""Constants shared by every feature function."""
FEATURE_VERSION = 2   # 2: R-peak detector band 8-25 Hz + abs envelope (RR + ECG-quality features changed)

IMU_FS = 50                 # Hz, waist IMU
IMU_WINDOW_S = 2.56         # 128 samples, 50 % overlap (docs/04)
ECG_FS = 250                # Hz, AD8232 via ADS1115
ECG_QUALITY_WINDOW_S = 5.0
RHYTHM_WINDOW_S = 30.0
FALL_WINDOW_S = 4.0         # 2 s before / 2 s after the impact trigger
EPS = 1e-9


In [ ]:
%%writefile senseband_features/imu.py
"""Waist-IMU window features for the activity / posture model (M1).

Input is accelerometer data in g in the BAND frame: x forward, y left, z up when standing (so gravity is
+z standing, mostly +/-x or y when lying). Dataset loaders rotate foreign sensors into this frame first.
"""
from __future__ import annotations

import numpy as np
from scipy.stats import kurtosis, skew

from .common import EPS, IMU_FS

_BANDS = ((0.3, 1.0), (1.0, 3.0), (3.0, 6.0), (6.0, 25.0))

IMU_FEATURES = (
    ["mean_x", "mean_y", "mean_z", "mean_norm", "tilt_z_deg", "abs_mean_x", "abs_mean_y", "abs_mean_z"]
    + ["std_x", "std_y", "std_z", "std_norm", "mean_abs_norm_dev", "min_norm", "max_norm", "range_norm"]
    + ["p10_norm", "p90_norm", "kurt_norm", "skew_norm", "sma", "zcr_norm", "mean_jerk"]
    + ["band_0p3_1", "band_1_3", "band_3_6", "band_6_25", "dom_freq", "dom_peak_ratio", "spec_entropy"]
    + ["corr_xy", "corr_xz", "corr_yz", "z_dom_freq", "autocorr_peak", "autocorr_lag_s"]
)


def _corr(a: np.ndarray, b: np.ndarray) -> float:
    sa, sb = a.std(), b.std()
    return 0.0 if sa < 1e-6 or sb < 1e-6 else float(np.corrcoef(a, b)[0, 1])


def imu_window_features(acc_g: np.ndarray, fs: float = IMU_FS) -> np.ndarray:
    """Feature vector (len(IMU_FEATURES),) for one (n, 3) accelerometer window."""
    a = np.asarray(acc_g, dtype=np.float64)
    if a.ndim != 2 or a.shape[1] != 3 or len(a) < 16:
        raise ValueError("expected an (n>=16, 3) accelerometer window")
    mean = a.mean(axis=0)
    norm = np.linalg.norm(a, axis=1)
    mnorm = float(np.linalg.norm(mean))
    tilt = float(np.degrees(np.arccos(np.clip(mean[2] / (mnorm + EPS), -1.0, 1.0))))
    dyn = a - mean                                   # gravity removed (per-window mean)
    dn = norm - norm.mean()
    std = a.std(axis=0)
    sma = float(np.mean(np.abs(dyn).sum(axis=1)))
    zcr = float(np.mean(np.diff(np.signbit(dn).astype(np.int8)) != 0))
    jerk = float(np.mean(np.linalg.norm(np.diff(a, axis=0), axis=1)) * fs)

    spec = np.abs(np.fft.rfft(dn * np.hanning(len(dn)))) ** 2
    freqs = np.fft.rfftfreq(len(dn), 1.0 / fs)
    tot = float(spec[freqs >= 0.3].sum()) + EPS
    bands = [float(spec[(freqs >= lo) & (freqs < hi)].sum() / tot) for lo, hi in _BANDS]
    rng = (freqs >= 0.5) & (freqs <= 5.0)
    if rng.any() and spec[rng].max() > 0:
        k = int(np.argmax(np.where(rng, spec, 0)))
        dom, peak_ratio = float(freqs[k]), float(spec[k] / tot)
    else:
        dom, peak_ratio = 0.0, 0.0
    p = spec[freqs >= 0.3] / tot
    entropy = float(-(p * np.log(p + EPS)).sum() / np.log(max(len(p), 2)))
    zspec = np.abs(np.fft.rfft((a[:, 2] - a[:, 2].mean()) * np.hanning(len(a)))) ** 2
    zr = (freqs >= 0.5) & (freqs <= 5.0)
    zdom = float(freqs[int(np.argmax(np.where(zr, zspec, 0)))]) if zr.any() and zspec[zr].max() > 0 else 0.0

    ac = np.correlate(dn, dn, mode="full")[len(dn) - 1:]
    ac = ac / (ac[0] + EPS)
    lo_lag, hi_lag = int(0.25 * fs), min(int(1.5 * fs), len(ac) - 1)
    if hi_lag > lo_lag:
        j = lo_lag + int(np.argmax(ac[lo_lag:hi_lag + 1]))
        acp, acl = float(ac[j]), j / fs
    else:
        acp, acl = 0.0, 0.0

    out = [*mean, mnorm, tilt, *np.abs(mean),
           *std, float(norm.std()), float(np.mean(np.abs(norm - 1.0))), float(norm.min()), float(norm.max()),
           float(np.ptp(norm)), float(np.percentile(norm, 10)), float(np.percentile(norm, 90)),
           float(kurtosis(norm)) if norm.std() > 1e-6 else 0.0, float(skew(norm)) if norm.std() > 1e-6 else 0.0,
           sma, zcr, jerk, *bands, dom, peak_ratio, entropy,
           _corr(dyn[:, 0], dyn[:, 1]), _corr(dyn[:, 0], dyn[:, 2]), _corr(dyn[:, 1], dyn[:, 2]), zdom, acp, acl]
    v = np.asarray(out, dtype=np.float32)
    assert len(v) == len(IMU_FEATURES), (len(v), len(IMU_FEATURES))
    return np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
%%writefile senseband_features/fall.py
"""Fall-vs-ADL features for a 4 s waist-IMU window centred on an impact trigger (model M2)."""
from __future__ import annotations

import numpy as np

from .common import EPS, FALL_WINDOW_S, IMU_FS

FALL_FEATURES = [
    "peak_g", "peak_time_s", "min_g_before_peak", "freefall_s", "impact_width_s", "jerk_peak",
    "pre_std", "post_std", "post_mean_dev", "post_std_ratio", "orient_change_deg", "z_drop",
    "energy_post", "n_over_2g",
]


def fall_window_features(acc_g: np.ndarray, fs: float = IMU_FS) -> np.ndarray:
    """acc_g: (n, 3) in g, ~FALL_WINDOW_S long with the trigger near the middle. Returns (len(FALL_FEATURES),)."""
    a = np.asarray(acc_g, dtype=np.float64)
    if a.ndim != 2 or a.shape[1] != 3 or len(a) < int(fs):
        raise ValueError("expected an (n>=fs, 3) accelerometer window")
    n = len(a)
    norm = np.linalg.norm(a, axis=1)
    k = int(np.argmax(norm))                                   # impact = strongest sample in the window
    t_peak = k / fs
    before = norm[max(0, k - int(1.0 * fs)):k] if k > 0 else norm[:1]
    ff = float(before.min()) if len(before) else 1.0
    ff_len = float(np.sum(before < 0.5) / fs) if len(before) else 0.0
    width = float(np.sum(norm[max(0, k - int(0.5 * fs)):k + int(0.5 * fs)] > 0.5 * (norm[k] + 1.0)) / fs)
    jerk = float(np.max(np.linalg.norm(np.diff(a, axis=0), axis=1)) * fs) if n > 1 else 0.0
    pre = norm[:max(1, k - int(0.5 * fs))]
    post = norm[min(n - 1, k + int(0.5 * fs)):]
    if len(post) < 3:
        post = norm[-3:]
    g_pre = a[max(0, k - int(1.5 * fs)):max(1, k - int(0.5 * fs))].mean(axis=0)
    g_post = a[min(n - 1, k + int(0.5 * fs)):].mean(axis=0) if k + int(0.5 * fs) < n else a[-3:].mean(axis=0)
    cosang = float(np.dot(g_pre, g_post) / (np.linalg.norm(g_pre) * np.linalg.norm(g_post) + EPS))
    orient = float(np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0))))
    out = [float(norm[k]), t_peak, ff, ff_len, width, jerk, float(pre.std()), float(post.std()),
           float(np.mean(np.abs(post - 1.0))), float(post.std() / (pre.std() + 0.01)), orient,
           float(g_pre[2] - g_post[2]), float(np.sum((post - 1.0) ** 2) / fs), float(np.sum(norm > 2.0))]
    v = np.asarray(out, dtype=np.float32)
    assert len(v) == len(FALL_FEATURES)
    return np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)


def window_samples(fs: float = IMU_FS) -> int:
    return int(FALL_WINDOW_S * fs)


In [ ]:
%%writefile senseband_features/hr.py
"""Features for the personal expected-heart-rate model (M5)."""
from __future__ import annotations

import numpy as np

ACTIVITY_CODES = {"lying": 0, "sleeping": 0, "sitting": 1, "standing": 2, "walking": 3, "stairs": 4,
                  "exercising": 5, "running": 6, "unknown": 2}

EXPECTED_HR_FEATURES = ["activity", "intensity", "cadence_hz", "load_5min", "tod_sin", "tod_cos", "temp_c", "beta_blocker"]


def expected_hr_features(activity: str, intensity: float, cadence_hz: float, load_5min: float,
                         hour_of_day: float, temp_c: float | None = None, beta_blocker: bool = False) -> np.ndarray:
    """intensity 0..1 (movement), load_5min = mean intensity over the last 5 minutes (exertion history)."""
    ang = 2 * np.pi * (hour_of_day % 24.0) / 24.0
    v = [ACTIVITY_CODES.get(activity, 2), float(intensity), float(cadence_hz), float(load_5min),
         float(np.sin(ang)), float(np.cos(ang)), 36.5 if temp_c is None else float(temp_c), 1.0 if beta_blocker else 0.0]
    return np.asarray(v, dtype=np.float32)


In [ ]:
%%writefile senseband_features/ecg.py
"""ECG features: R peaks, RR-interval features for rhythm screening (M4) and signal-quality features (M3).

Single lead, any sampling rate >= 125 Hz (training resamples PhysioNet data to 250 Hz, like the edge ADC).
RR features follow the CinC-2017 literature: variability, entropy, Poincare, turning points.
"""
from __future__ import annotations

import numpy as np
from scipy.signal import butter, find_peaks, sosfiltfilt, welch
from scipy.stats import kurtosis, skew

from .common import ECG_FS, EPS

RR_FEATURES = [
    "n_beats", "mean_rr", "hr_bpm", "sdnn", "rmssd", "pnn50", "pnn20", "cv_rr", "range_rr", "iqr_rr",
    "mad_diff", "sd1", "sd2", "sd_ratio", "sampen", "cosen", "turning_point_ratio", "shannon_entropy",
    "frac_irregular", "frac_short", "frac_long", "max_over_median",
]

ECG_QUALITY_FEATURES = [
    "std", "ptp", "kurtosis", "skew", "flat_frac", "clip_frac", "qrs_band_ratio", "baseline_ratio",
    "hf_ratio", "n_peaks", "rr_cv", "template_corr", "template_corr_min", "peak_amp_ratio", "diff_std_ratio",
]


# ------------------------------------------------------------------ R peaks
def bandpass(x: np.ndarray, fs: float, lo: float = 8.0, hi: float = 25.0) -> np.ndarray:
    sos = butter(2, [lo, min(hi, fs / 2 - 1)], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, x)


def r_peaks(x: np.ndarray, fs: float = ECG_FS) -> np.ndarray:
    """R-peak sample indices: 8-25 Hz band-pass, slope+amplitude envelope, adaptive threshold, refractory 0.25 s.

    Band and envelope were chosen on CinC-2017 by subject-wise AF F1 (see metrics.json history in CHANGELOG)."""
    x = np.asarray(x, dtype=np.float64)
    if len(x) < fs * 2 or not np.isfinite(x).all() or np.ptp(x) < 1e-6:
        return np.array([], dtype=int)
    y = bandpass(x - np.median(x), fs)
    env = np.convolve(np.abs(np.diff(y, prepend=y[0])) + np.abs(y), np.ones(int(0.08 * fs)) / int(0.08 * fs), "same")
    thr = 0.3 * np.percentile(env, 99)
    cand, _ = find_peaks(env, height=thr, distance=int(0.25 * fs))
    if len(cand) == 0:
        return cand
    half = int(0.06 * fs)
    ref = np.array([max(0, c - half) + int(np.argmax(np.abs(y[max(0, c - half):c + half + 1]))) for c in cand])
    return np.unique(ref)


def rr_intervals(peaks: np.ndarray, fs: float = ECG_FS) -> np.ndarray:
    """Seconds between beats; physiologically impossible gaps (< 0.2 s or > 3 s) are dropped."""
    rr = np.diff(np.asarray(peaks, dtype=np.float64)) / fs
    return rr[(rr >= 0.2) & (rr <= 3.0)]


# ------------------------------------------------------------------ RR features
def _sampen(x: np.ndarray, m: int = 2, r_frac: float = 0.2) -> float:
    n = len(x)
    if n < m + 3:
        return 0.0
    r = r_frac * x.std()
    if r < 1e-9:
        return 0.0

    def phi(mm: int) -> float:
        t = np.lib.stride_tricks.sliding_window_view(x, mm)[: n - m]
        d = np.max(np.abs(t[:, None, :] - t[None, :, :]), axis=2)
        return float((np.sum(d <= r) - len(t)) / 2.0)

    a, b = phi(m + 1), phi(m)
    return float(-np.log(a / b)) if a > 0 and b > 0 else float(np.log(max(n - m, 2)))


def rr_features(rr: np.ndarray) -> np.ndarray:
    """Feature vector (len(RR_FEATURES),) from RR intervals in seconds (any count; short input is safe)."""
    r = np.asarray(rr, dtype=np.float64)
    n = len(r)
    if n < 4:
        v = np.zeros(len(RR_FEATURES), dtype=np.float32)
        v[0] = n + 1
        return v
    if n > 120:
        r = r[:120]
        n = 120
    mean = r.mean()
    d = np.diff(r)
    sd1 = float(np.std(d) / np.sqrt(2))
    sd2 = float(np.sqrt(max(2 * r.std() ** 2 - sd1 ** 2, 0.0)))
    se = _sampen(r)
    cosen = se + float(np.log(2 * 0.2 * r.std() + EPS)) - float(np.log(mean))
    tp = np.sum((r[1:-1] > r[:-2]) & (r[1:-1] > r[2:])) + np.sum((r[1:-1] < r[:-2]) & (r[1:-1] < r[2:]))
    tpr = float(tp / (2.0 / 3.0 * (n - 2))) if n > 2 else 0.0
    hist, _ = np.histogram(r, bins=np.linspace(0.3, 2.0, 18))
    p = hist[hist > 0] / max(hist.sum(), 1)
    out = [n + 1, mean, 60.0 / mean, r.std(), float(np.sqrt(np.mean(d ** 2))), float(np.mean(np.abs(d) > 0.05)),
           float(np.mean(np.abs(d) > 0.02)), r.std() / mean, float(np.ptp(r)),
           float(np.subtract(*np.percentile(r, [75, 25]))), float(np.median(np.abs(d))), sd1, sd2,
           sd1 / (sd2 + EPS), se, cosen, tpr, float(-(p * np.log(p)).sum()), float(np.mean(np.abs(d) > 0.1 * mean)),
           float(np.mean(r < 0.8 * mean)), float(np.mean(r > 1.2 * mean)), float(r.max() / np.median(r))]
    v = np.asarray(out, dtype=np.float32)
    assert len(v) == len(RR_FEATURES)
    return np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)


# ------------------------------------------------------------------ signal quality
def ecg_quality_features(x: np.ndarray, fs: float = ECG_FS) -> np.ndarray:
    """Quality features for a ~5 s ECG window (mV or any linear scale: features are scale-free)."""
    x = np.asarray(x, dtype=np.float64)
    if len(x) < fs * 2:
        raise ValueError("need at least 2 s of ECG")
    x = np.nan_to_num(x)
    xc = x - np.median(x)
    std, ptp = float(xc.std()), float(np.ptp(xc))
    d = np.diff(xc)
    flat = float(np.mean(np.abs(d) < 1e-3 * (ptp + EPS)))
    clip = float(np.mean(np.abs(xc) >= 0.98 * (np.max(np.abs(xc)) + EPS))) if ptp > 0 else 1.0
    f, pxx = welch(xc, fs=fs, nperseg=min(len(xc), int(fs * 2)))
    tot = float(pxx[(f >= 0.5) & (f <= 40)].sum()) + EPS
    qrs = float(pxx[(f >= 5) & (f <= 15)].sum() / tot)
    base = float(pxx[f < 0.5].sum() / (pxx[f <= 40].sum() + EPS))
    hf = float(pxx[(f > 40)].sum() / (pxx.sum() + EPS))
    peaks = r_peaks(x, fs)
    rr = rr_intervals(peaks, fs)
    cv = float(rr.std() / rr.mean()) if len(rr) >= 2 else 1.0
    tcorr = tmin = 0.0
    amp_ratio = 0.0
    if len(peaks) >= 3:
        half = int(0.25 * fs)
        beats = np.asarray([xc[p - half:p + half] for p in peaks if p - half >= 0 and p + half <= len(xc)])
        if len(beats) >= 3:
            tpl = np.median(beats, axis=0)
            cs = [np.corrcoef(b, tpl)[0, 1] for b in beats if b.std() > 1e-9 and tpl.std() > 1e-9]
            if cs:
                tcorr, tmin = float(np.mean(cs)), float(np.min(cs))
        amps = np.abs(xc[peaks])
        amp_ratio = float(amps.max() / (np.median(amps) + EPS))
    diff_ratio = float(np.std(d) / (std + EPS))
    out = [std, ptp, float(kurtosis(xc)) if std > 1e-9 else 0.0, float(skew(xc)) if std > 1e-9 else 0.0, flat, clip,
           qrs, base, hf, float(len(peaks)), cv, tcorr, tmin, amp_ratio, diff_ratio]
    v = np.asarray(out, dtype=np.float32)
    assert len(v) == len(ECG_QUALITY_FEATURES)
    return np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
%%writefile senseband_train/__init__.py
"""Training pipeline: dataset download/convert, windows, LightGBM training, ONNX export, metrics, benchmark."""

from senseband_features import FEATURE_VERSION  # noqa: E402,F401  (training and edge share one definition)


In [ ]:
%%writefile senseband_train/core.py
"""Training utilities: LightGBM, subject-wise cross-validation, ONNX export + parity, latency, metrics."""
from __future__ import annotations

import json
import platform
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np

MODELS_DIR = Path.cwd() / "models"
MAX_MODEL_BYTES = 5 * 1024 * 1024        # CLAUDE.md: models < 5 MB each
MAX_LATENCY_MS = 50.0                    # CLAUDE.md: inference < 50 ms per window (on the Pi 3)

CLASSIFIER_PARAMS: dict[str, Any] = dict(
    n_estimators=200, learning_rate=0.06, num_leaves=24, min_child_samples=15, subsample=0.8,
    subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=4, verbose=-1, random_state=0,
)


def make_classifier(**over: Any):  # noqa: ANN201
    from lightgbm import LGBMClassifier
    return LGBMClassifier(**{**CLASSIFIER_PARAMS, "class_weight": "balanced", **over})


def make_regressor(**over: Any):  # noqa: ANN201
    from lightgbm import LGBMRegressor
    return LGBMRegressor(**{**CLASSIFIER_PARAMS, **over})


def group_folds(groups: np.ndarray, n_splits: int = 5):  # noqa: ANN201
    """Subject-wise folds: no person ever appears in both train and test."""
    from sklearn.model_selection import GroupKFold
    n = min(n_splits, len(set(groups.tolist())))
    return list(GroupKFold(n_splits=n).split(np.zeros(len(groups)), groups=groups))


def cross_val_classify(X: np.ndarray, y: np.ndarray, groups: np.ndarray, n_splits: int = 5,
                       **over: Any) -> dict[str, Any]:
    """Out-of-fold predictions with subject-wise folds -> honest generalisation metrics."""
    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
    classes = sorted(set(y.tolist()))
    pred = np.empty(len(y), dtype=object)
    fold_f1 = []
    for tr, te in group_folds(groups, n_splits):
        m = make_classifier(**over).fit(X[tr], y[tr])
        pred[te] = m.predict(X[te])
        fold_f1.append(float(f1_score(y[te], pred[te], average="macro", labels=classes, zero_division=0)))
    f1s = f1_score(y, pred, average=None, labels=classes, zero_division=0)
    return {"scheme": f"GroupKFold({n_splits}) by subject/recording", "n": int(len(y)),
            "n_groups": int(len(set(groups.tolist()))),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=classes, zero_division=0)),
            "accuracy": float(accuracy_score(y, pred)),
            "per_class_f1": {c: float(v) for c, v in zip(classes, f1s, strict=True)},
            "confusion": {"labels": classes, "matrix": confusion_matrix(y, pred, labels=classes).tolist()},
            "fold_macro_f1": fold_f1, "oof_pred": pred}


def export_onnx(model: Any, n_features: int, path: Path, regressor: bool = False) -> Path:
    from onnxmltools import convert_lightgbm
    from onnxmltools.convert.common.data_types import FloatTensorType
    kw = {} if regressor else {"zipmap": False}
    onx = convert_lightgbm(model, initial_types=[("features", FloatTensorType([None, n_features]))],
                           target_opset=14, **kw)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(onx.SerializeToString())
    return path


def onnx_session(path: Path):  # noqa: ANN201
    import onnxruntime as ort
    so = ort.SessionOptions()
    so.intra_op_num_threads = so.inter_op_num_threads = 1       # the Pi has few cores; keep it predictable
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])


def onnx_predict(sess: Any, X: np.ndarray) -> np.ndarray:
    """Class probabilities (n, k) or regression values (n,)."""
    out = sess.run(None, {"features": X.astype(np.float32)})
    return np.asarray(out[1] if len(out) > 1 else out[0])


def verify_parity(model: Any, sess: Any, X: np.ndarray, regressor: bool = False) -> float:
    ref = model.predict(X) if regressor else model.predict_proba(X)
    got = onnx_predict(sess, X).reshape(np.shape(ref))
    return float(np.max(np.abs(np.asarray(ref, dtype=np.float64) - got)))


def latency_ms(sess: Any, n_features: int, runs: int = 300) -> dict[str, float]:
    x = np.random.default_rng(0).normal(size=(1, n_features)).astype(np.float32)
    for _ in range(20):
        sess.run(None, {"features": x})
    ts = []
    for _ in range(runs):
        t = time.perf_counter()
        sess.run(None, {"features": x})
        ts.append((time.perf_counter() - t) * 1000)
    return {"median": round(float(np.median(ts)), 3), "p95": round(float(np.percentile(ts, 95)), 3)}


def libs() -> dict[str, str]:
    import lightgbm
    import onnxruntime
    import sklearn
    return {"python": platform.python_version(), "lightgbm": lightgbm.__version__, "sklearn": sklearn.__version__,
            "onnxruntime": onnxruntime.__version__, "numpy": np.__version__}


def write_report(name: str, report: dict[str, Any], out_dir: Path = MODELS_DIR) -> None:
    """Merge one model's report into metrics.json (other models' entries are kept)."""
    path = out_dir / "metrics.json"
    allm = json.loads(path.read_text()) if path.exists() else {}
    report = {k: v for k, v in report.items()}
    report["trained_at"] = datetime.now(UTC).isoformat(timespec="seconds")
    report["libs"] = libs()
    allm[name] = report
    out_dir.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(allm, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))


def finish_classifier(name: str, X: np.ndarray, y: np.ndarray, cv: dict[str, Any], features: list[str], data: dict,
                      target: dict[str, Any], out_dir: Path = MODELS_DIR, extra: dict | None = None) -> dict[str, Any]:
    """Fit on everything, export ONNX, verify parity + size + latency, write the metrics entry."""
    from . import FEATURE_VERSION  # noqa: PLC0415
    model = make_classifier().fit(X, y)
    path = export_onnx(model, X.shape[1], out_dir / f"{name}.onnx")
    sess = onnx_session(path)
    idx = np.random.default_rng(0).choice(len(X), size=min(500, len(X)), replace=False)
    diff = verify_parity(model, sess, X[idx])
    size, lat = path.stat().st_size, latency_ms(sess, X.shape[1])
    assert diff < 1e-3, f"{name}: ONNX output differs from LightGBM by {diff}"
    assert size < MAX_MODEL_BYTES, f"{name}: {size} bytes exceeds the 5 MB budget"
    cv = {k: v for k, v in cv.items() if k != "oof_pred"}
    rep = {"name": name, "algorithm": "lightgbm", "feature_version": FEATURE_VERSION, "features": features,
           "classes": [str(c) for c in model.classes_], "data": data, "cv": cv, "target": target,
           "onnx": {"file": path.name, "bytes": size, "parity_max_abs_diff": diff},
           "latency_ms_this_machine": lat, "latency_budget_ms": MAX_LATENCY_MS, **(extra or {})}
    write_report(name, rep, out_dir)
    return rep


In [ ]:
%%writefile senseband_train/datasets.py
"""Dataset download + safe extraction + loaders (all returning data in the BAND frame / 250 Hz).

Downloaded archives are UNTRUSTED data: each goes in its own directory, extraction refuses path traversal and
absolute paths and caps total size, and loaders only ever read numeric files from the extracted tree.

Real data used (see docs/04_ML_MODELS.md):
  uci_har   30 people, waist-worn phone, 50 Hz                  -> M1 (activity / posture)
  pamap2    9 people, chest IMU + heart rate, 100 Hz             -> M1 (running etc.), M5 (expected HR)
  cinc2017  8,528 single-lead ECG recordings, AF / normal / other / noisy -> M3 (quality), M4 (rhythm)
  mitbih    (optional) MIT-BIH arrhythmia, via wfdb              -> script only, not used by default
  sisfall   falls + ADLs: the download host was unreachable when this was written; download by hand and use
            `load_sisfall(dir)` (M2 is trained on SIMULATED falls until then, and is flagged as such)
"""
from __future__ import annotations

import csv
import io
import shutil
import urllib.request
import zipfile
from collections.abc import Iterator
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from scipy.signal import resample_poly

ROOT = Path.cwd() / "data"
RAW, OUT = ROOT / "raw", ROOT

URLS = {
    "uci_har": "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "pamap2": "https://archive.ics.uci.edu/static/public/231/pamap2+physical+activity+monitoring.zip",
    "cinc2017": "https://physionet.org/files/challenge-2017/1.0.0/training2017.zip",
}
MAX_EXTRACT_BYTES = 6 * 1024 ** 3


def download(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 1024:
        return dest
    with urllib.request.urlopen(url, timeout=60) as r, open(dest, "wb") as f:   # noqa: S310 - fixed https URLs above
        shutil.copyfileobj(r, f, 1 << 20)
    return dest


def safe_extract(zip_path: Path, dest: Path) -> list[Path]:
    """Extract without trusting member names. Raises on traversal, absolute paths or zip bombs."""
    dest = dest.resolve()
    dest.mkdir(parents=True, exist_ok=True)
    out: list[Path] = []
    with zipfile.ZipFile(zip_path) as z:
        if sum(i.file_size for i in z.infolist()) > MAX_EXTRACT_BYTES:
            raise ValueError("archive too large to extract safely")
        for info in z.infolist():
            target = (dest / info.filename).resolve()
            if dest != target and dest not in target.parents:
                raise ValueError(f"unsafe path in archive: {info.filename!r}")
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, 1 << 20)
            out.append(target)
    return out


def fetch(name: str) -> Path:
    """Download (if needed) and extract one dataset into ml/data/<name>/. Returns that directory."""
    final = OUT / name
    if final.exists() and any(final.iterdir()):
        return final
    z = download(URLS[name], RAW / name / f"{name}.zip")
    safe_extract(z, final)
    for inner in list(final.glob("*.zip")):                 # UCI wraps the real archive in another zip
        safe_extract(inner, final)
    return final


# ----------------------------------------------------------------------------------------------- helpers
def align_up(acc: np.ndarray, up: np.ndarray) -> np.ndarray:
    """Rotate a dataset's accelerometer frame so that `up` (gravity direction when standing) becomes +z.
    One rotation per DATASET (not per person), so posture classes keep realistic variation."""
    up = up / np.linalg.norm(up)
    z = np.array([0.0, 0.0, 1.0])
    v, c = np.cross(up, z), float(up @ z)
    if np.linalg.norm(v) < 1e-9:
        return acc if c > 0 else acc * np.array([1, -1, -1])
    k = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    r = np.eye(3) + k + k @ k * (1 / (1 + c))
    return acc @ r.T


# ----------------------------------------------------------------------------------------------- UCI HAR
HAR_CLASSES = {1: "walking", 2: "stairs", 3: "stairs", 4: "sitting", 5: "standing", 6: "lying"}


def load_uci_har(root: Path | None = None) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """(windows (n,128,3) in g, band frame; labels (str); subject ids) for all 30 subjects, 50 Hz."""
    base = next((root or OUT / "uci_har").rglob("UCI HAR Dataset"))
    xs, ys, ss = [], [], []
    for split in ("train", "test"):
        sig = np.stack([np.loadtxt(base / split / "Inertial Signals" / f"total_acc_{a}_{split}.txt")
                        for a in "xyz"], axis=-1)
        xs.append(sig)
        ys.append(np.loadtxt(base / split / f"y_{split}.txt", dtype=int))
        ss.append(np.loadtxt(base / split / f"subject_{split}.txt", dtype=int))
    x, y, s = np.concatenate(xs), np.concatenate(ys), np.concatenate(ss)
    up = x[y == 5].reshape(-1, 3).mean(axis=0)               # standing windows define "up"
    x = np.stack([align_up(w, up) for w in x])
    return x.astype(np.float32), np.array([HAR_CLASSES[i] for i in y]), s


# ----------------------------------------------------------------------------------------------- PAMAP2
PAMAP_CLASSES = {1: "lying", 2: "sitting", 3: "standing", 4: "walking", 5: "running", 6: "exercising",
                 7: "exercising", 9: "sitting", 10: "sitting", 11: "sitting", 12: "stairs", 13: "stairs",
                 17: "standing", 18: "standing", 20: "exercising", 24: "exercising"}


def pamap_subject_files(root: Path | None = None) -> list[Path]:
    return sorted((root or OUT / "pamap2").rglob("subject10*.dat"))


def load_pamap2(root: Path | None = None, win: int = 128, hop: int = 128
                ) -> dict[str, np.ndarray]:
    """Chest IMU (+/-16 g channel) windows at 50 Hz in the band frame, with mean heart rate per window.

    Returns arrays: x (n,win,3) g, y (str), subject (int), hr (n,) bpm (nan if missing), t (n,) window start s
    (seconds into that subject's recording, for the exertion-history feature)."""
    files = pamap_subject_files(root)
    if not files:
        raise FileNotFoundError("PAMAP2 not found: run `python -m senseband_train.prepare pamap2`")
    raw: list[tuple[int, np.ndarray]] = []
    for f in files:
        d = np.genfromtxt(f, delimiter=" ", usecols=(0, 1, 2, 21, 22, 23))
        raw.append((int(f.stem[-3:]), d))
    # one "up" for the dataset, from all standing samples
    stand = np.concatenate([d[d[:, 1] == 3][:, 3:6] for _, d in raw])
    up = np.nanmean(stand, axis=0)
    xs, ys, ss, hrs, ts = [], [], [], [], []
    for subj, d in raw:
        d = d[d[:, 1] > 0]                                      # drop transient (0)
        acc = d[:, 3:6] / 9.80665
        ok = np.isfinite(acc).all(axis=1)
        acc, lab, hr = acc[ok], d[ok, 1].astype(int), d[ok, 2]
        t0 = d[ok, 0]
        acc = align_up(acc, up / 9.80665)
        acc = resample_poly(acc, 1, 2, axis=0)                  # 100 Hz -> 50 Hz
        lab, hr, t0 = lab[::2][: len(acc)], hr[::2][: len(acc)], t0[::2][: len(acc)]
        for i in range(0, len(acc) - win, hop):
            seg = lab[i:i + win]
            u, c = np.unique(seg, return_counts=True)
            top = int(u[np.argmax(c)])
            if c.max() < 0.9 * win or top not in PAMAP_CLASSES:
                continue
            xs.append(acc[i:i + win].astype(np.float32))
            ys.append(PAMAP_CLASSES[top])
            ss.append(subj)
            h = hr[i:i + win]
            hrs.append(float(np.nanmean(h)) if np.isfinite(h).any() else np.nan)
            ts.append(float(t0[i]))
    return {"x": np.stack(xs), "y": np.array(ys), "subject": np.array(ss), "hr": np.array(hrs), "t": np.array(ts)}


# ----------------------------------------------------------------------------------------------- CinC 2017
CINC_LABELS = {"N": "normal", "A": "af", "O": "other", "~": "noisy"}


def iter_cinc2017(root: Path | None = None, fs_out: int = 250, limit: int | None = None
                  ) -> Iterator[tuple[str, str, np.ndarray]]:
    """Yield (record id, label, ECG in mV resampled to fs_out) for every labelled recording."""
    base = next((root or OUT / "cinc2017").rglob("REFERENCE.csv")).parent
    with open(base / "REFERENCE.csv", newline="") as f:
        rows = list(csv.reader(f))
    for n, (rid, lab) in enumerate(rows):
        if limit and n >= limit:
            break
        m = loadmat(base / f"{rid}.mat")["val"].astype(np.float64).ravel() / 1000.0
        yield rid, CINC_LABELS[lab], resample_poly(m, fs_out // 50, 300 // 50)


# ----------------------------------------------------------------------------------------------- optional
def fetch_mitbih(dest: Path | None = None) -> Path:   # pragma: no cover - needs network
    import wfdb
    d = dest or OUT / "mitbih"
    wfdb.dl_database("mitdb", str(d))
    return d


def load_sisfall(root: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:   # pragma: no cover - manual data
    """SisFall (waist ADXL345 +/-16 g at 200 Hz): files like `SA01/D01_SA01_R01.txt` / `F01_...`.
    Returns 4 s windows at 50 Hz around the strongest sample, label 'fall' (F*) / 'adl' (D*), subject ids."""
    from scipy.signal import resample_poly as rp
    xs, ys, ss = [], [], []
    for f in sorted(root.rglob("*.txt")):
        if f.name[0] not in "DF" or "_" not in f.name:
            continue
        txt = f.read_text().replace(";", "")
        d = np.loadtxt(io.StringIO(txt), delimiter=",", usecols=(0, 1, 2))
        acc = d * (32.0 / 2 ** 13)                              # ADXL345 counts -> g
        acc = rp(acc, 1, 4, axis=0)                             # 200 -> 50 Hz
        k = int(np.argmax(np.linalg.norm(acc, axis=1)))
        lo = max(0, min(k - 100, len(acc) - 200))
        xs.append(acc[lo:lo + 200])
        ys.append("fall" if f.name[0] == "F" else "adl")
        ss.append(int("".join(c for c in f.parent.name if c.isdigit()) or 0))
    return np.stack(xs).astype(np.float32), np.array(ys), np.array(ss)


# ----------------------------------------------------------------------------------------------- own data
def load_recordings(directory: Path, win: int = 128, hop: int = 128) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Windows from 'Record training data' files (edge/senseband_edge/record.py).

    Returns (windows (n, win, 3) g at 50 Hz, labels from the file names/metadata, group ids = file stems)."""
    xs, ys, gs = [], [], []
    for f in sorted(directory.glob("*.npz")):
        z = np.load(f, allow_pickle=False)
        acc = z["acc_g"].astype(np.float32)
        label = str(z["meta_label"]) if "meta_label" in z.files else f.stem.split("_", 1)[-1]
        for i in range(0, len(acc) - win, hop):
            xs.append(acc[i:i + win])
            ys.append(label)
            gs.append(f.stem)
    if not xs:
        return np.zeros((0, win, 3), np.float32), np.array([]), np.array([])
    return np.stack(xs), np.array(ys), np.array(gs)


In [ ]:
%%writefile senseband_train/sim_data.py
"""Training data generated by the hardware simulator (edge/senseband_edge/hw/sim).

SIMULATED data only proves the pipeline and covers cases no public dataset has (flat ECG, lead-off). Anything
trained on it is flagged `provenance: simulated` in metrics.json and must not be read as real-world accuracy.
"""
from __future__ import annotations

import copy

import numpy as np
from senseband_features import fall_window_features
from senseband_features.common import ECG_FS, FALL_WINDOW_S, IMU_FS

IMU_N = int(FALL_WINDOW_S * IMU_FS)


def _sim(segments: list[dict], cfg: dict, seed: int):  # noqa: ANN202
    from senseband_edge.hw.sim import Simulator
    from senseband_edge.hw.sim.scenario import Timeline
    tl = Timeline({"name": "train", "seed": seed, "segments": segments}, cfg, seed)
    return Simulator(tl, cfg, seed)


def _waist(sim, seconds: float) -> np.ndarray:  # noqa: ANN001
    return np.concatenate([f.imu["waist"].acc_g for f in sim.frames(seconds)])


def _random_cfg(rng: np.random.Generator) -> dict:
    from senseband_edge.hw.sim import load_config
    cfg = copy.deepcopy(load_config())
    d = rng.normal(size=3)
    d[2] = abs(d[2]) * 0.5 + 0.1
    f = cfg["imu"]["fall"]
    f.update(direction=(d / np.linalg.norm(d)).tolist(), impact_g=float(rng.uniform(2.8, 6.0)),
             freefall_g=float(rng.uniform(0.05, 0.35)), freefall_s=float(rng.uniform(0.2, 0.5)),
             rotate_s=float(rng.uniform(0.4, 1.0)))
    return cfg


def _centre(acc: np.ndarray, k: int) -> np.ndarray:
    lo = int(np.clip(k - IMU_N // 2, 0, max(0, len(acc) - IMU_N)))
    return acc[lo:lo + IMU_N]


def fall_dataset(n_pos: int = 240, n_neg: int = 720, seed: int = 0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """(features, labels 'fall'/'adl', group ids). The group is the simulator seed (a stand-in for a person)."""
    rng = np.random.default_rng(seed)
    X, y, g = [], [], []

    def add(win: np.ndarray, label: str, grp: int) -> None:
        X.append(fall_window_features(win))
        y.append(label)
        g.append(grp)

    for i in range(n_pos):                                         # --- falls from walking / standing / sitting
        cfg = _random_cfg(rng)
        pre = rng.choice(["walking", "standing", "jogging", "sitting"])
        sim = _sim([{"duration_s": 6, "activity": pre},
                    {"duration_s": 10, "activity": pre, "events": [{"type": "fall", "at_s": 3.0, "then": "lying_still"}]}],
                   cfg, int(rng.integers(1, 10 ** 6)))
        acc = _waist(sim, 16)
        te = 6 + 3.0 + cfg["imu"]["fall"]["freefall_s"]
        add(_centre(acc, int(te * IMU_FS)), "fall", i % 40)
    kinds = ["jog", "walk", "sitdown", "stumble", "jump", "lie_slow", "stand_still"]
    for i in range(n_neg):                                         # --- activities of daily living
        kind = kinds[i % len(kinds)]
        cfg = _random_cfg(rng)
        seed_i = int(rng.integers(1, 10 ** 6))
        if kind == "jog":
            acc = _waist(_sim([{"duration_s": 12, "activity": "jogging"}], cfg, seed_i), 12)
            add(_centre(acc, int(np.argmax(np.linalg.norm(acc, axis=1)))), "adl", i % 40)
        elif kind == "walk":
            acc = _waist(_sim([{"duration_s": 12, "activity": "walking"}], cfg, seed_i), 12)
            add(_centre(acc, int(np.argmax(np.linalg.norm(acc, axis=1)))), "adl", i % 40)
        elif kind == "stand_still":
            acc = _waist(_sim([{"duration_s": 12, "activity": "standing"}], cfg, seed_i), 12)
            add(_centre(acc, 200), "adl", i % 40)
        elif kind == "lie_slow":                                    # lying down on a bed: orientation change, no impact
            acc = _waist(_sim([{"duration_s": 5, "activity": "standing"},
                               {"duration_s": 8, "activity": "lying_still", "trans_s": float(rng.uniform(2, 4))}], cfg, seed_i), 13)
            add(_centre(acc, int(rng.uniform(5.5, 8) * IMU_FS)), "adl", i % 40)
        else:                                                       # synthetic ADL impacts on a calm signal
            base = "walking" if kind == "stumble" else ("standing" if kind == "jump" else "standing")
            segs = [{"duration_s": 5, "activity": base}]
            if kind == "sitdown":
                segs.append({"duration_s": 8, "activity": "sitting", "trans_s": float(rng.uniform(0.8, 1.6))})
            else:
                segs.append({"duration_s": 8, "activity": base})
            acc = _waist(_sim(segs, cfg, seed_i), 13).copy()
            k = int(rng.uniform(5.2, 7) * IMU_FS)
            t = np.arange(len(acc)) / IMU_FS
            up = np.array([0.1, 0.0, 1.0])
            if kind == "sitdown":
                amp = rng.uniform(0.6, 1.8)
                acc += amp * np.exp(-0.5 * ((t - k / IMU_FS) / 0.06) ** 2)[:, None] * up
            elif kind == "stumble":
                ff = float(rng.uniform(0.1, 0.25))
                m = (t >= k / IMU_FS - ff) & (t < k / IMU_FS)
                acc[m] *= rng.uniform(0.4, 0.7)
                acc += rng.uniform(2.4, 3.5) * np.exp(-0.5 * ((t - k / IMU_FS) / 0.03) ** 2)[:, None] * up
            else:                                                   # jump and land, then stand
                ff = float(rng.uniform(0.25, 0.45))
                m = (t >= k / IMU_FS - ff) & (t < k / IMU_FS)
                acc[m] *= 0.1
                acc += rng.uniform(2.5, 4.0) * np.exp(-0.5 * ((t - k / IMU_FS) / 0.04) ** 2)[:, None] * up
            add(_centre(acc, k), "adl", i % 40)
    return np.stack(X), np.array(y), np.array(g)


def unusable_ecg_windows(n: int = 400, seed: int = 0) -> list[np.ndarray]:
    """5 s windows of ECG that carry no usable signal: flat line, mains hum (lead-off), saturation, pure noise."""
    rng = np.random.default_rng(seed)
    n_s = int(5 * ECG_FS)
    t = np.arange(n_s) / ECG_FS
    out = []
    for i in range(n):
        k = i % 5
        if k == 0:
            w = rng.normal(0, rng.uniform(0.0005, 0.02), n_s)                               # flat + sensor noise
        elif k == 1:
            w = rng.uniform(0.1, 0.6) * np.sin(2 * np.pi * 50 * t + rng.uniform(0, 6)) + rng.normal(0, 0.01, n_s)  # lead-off hum
        elif k == 2:
            w = np.clip(rng.normal(0, 2.0, n_s) + 0.5 * np.sin(2 * np.pi * 0.3 * t), -1, 1)   # clipped
        elif k == 3:
            w = rng.normal(0, rng.uniform(0.3, 1.0), n_s)                                     # white noise
        else:
            w = np.cumsum(rng.normal(0, 0.02, n_s))                                           # drifting baseline only
        out.append(w.astype(np.float32))
    return out


In [ ]:
%%writefile senseband_train/train_models.py
"""Train M1-M5 and write ml/models/*.onnx + metrics.json (python -m senseband_train)."""
from __future__ import annotations

import time
from pathlib import Path
from typing import Any

import numpy as np
from senseband_features import (
    ECG_QUALITY_FEATURES,
    EXPECTED_HR_FEATURES,
    FALL_FEATURES,
    FEATURE_VERSION,
    IMU_FEATURES,
    RR_FEATURES,
    ecg_quality_features,
    expected_hr_features,
    imu_window_features,
    r_peaks,
    rr_features,
    rr_intervals,
)
from senseband_features.common import ECG_FS

from . import core, datasets
from .core import MODELS_DIR

CACHE = datasets.ROOT / "cache"


def _cache(name: str, build):  # noqa: ANN001, ANN202
    CACHE.mkdir(parents=True, exist_ok=True)
    p = CACHE / f"{name}.npz"
    if p.exists():
        z = np.load(p, allow_pickle=True)
        return {k: z[k] for k in z.files}
    d = build()
    np.savez_compressed(p, **d)
    return d


def _f1_target(metric: str, value: float, got: float, real: bool) -> dict[str, Any]:
    return {"metric": metric, "value": value, "got": round(float(got), 4), "met": bool(got >= value),
            "validated_on_real_data": real}


# ------------------------------------------------------------------------------------------------- M1
def train_m1(out: Path = MODELS_DIR, quick: bool = False) -> dict[str, Any]:
    """Activity & posture: UCI HAR (waist phone, 30 people) + PAMAP2 chest IMU (9 people, incl. running)."""
    def build() -> dict[str, np.ndarray]:
        hx, hy, hs = datasets.load_uci_har()
        p = datasets.load_pamap2(hop=256 if quick else 128)
        wins = np.concatenate([hx, p["x"]])
        feats = np.stack([imu_window_features(w) for w in wins])
        labels = np.concatenate([hy, p["y"]])
        groups = np.concatenate([[f"har{s}" for s in hs], [f"pamap{s}" for s in p["subject"]]])
        src = np.array(["uci_har"] * len(hx) + ["pamap2"] * len(p["x"]))
        return {"X": feats, "y": labels, "g": groups, "src": src}
    d = _cache("m1_quick" if quick else "m1", build)
    X, y, g = d["X"], d["y"].astype(str), d["g"].astype(str)
    cv = core.cross_val_classify(X, y, g, 5)
    base = _rule_baseline_m1(X, y)
    data = {"sources": ["UCI HAR (waist smartphone, 30 subjects, 50 Hz)", "PAMAP2 (chest IMU, 9 subjects, 100->50 Hz)"],
            "provenance": "real", "n_windows": int(len(y)), "n_subjects": int(len(set(g.tolist()))),
            "window": "128 samples @ 50 Hz = 2.56 s",
            "class_counts": {c: int((y == c).sum()) for c in sorted(set(y.tolist()))},
            "notes": "Subject-wise CV. 'running' and 'exercising' come only from PAMAP2 (9 people); no separate "
                     "hold-out dataset. Sleeping is a rule on top (lying + still >= 2 min), not modelled."}
    rep = core.finish_classifier("m1_activity", X, y, cv, IMU_FEATURES, data,
                                 _f1_target("macro_f1", 0.90, cv["macro_f1"], True), out,
                                 extra={"rule_baseline_on_same_windows": base})
    return rep


def _rule_baseline_m1(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """The edge's rule-based classifier evaluated on the same feature rows (it only needs std / gravity)."""
    from sklearn.metrics import f1_score
    ix = {n: i for i, n in enumerate(IMU_FEATURES)}
    pred = []
    for r in X:
        lying = max(abs(r[ix["mean_x"]]), abs(r[ix["mean_y"]])) / max(r[ix["mean_norm"]], 1e-6) > 0.7
        std = r[ix["std_norm"]]
        if lying:
            pred.append("lying")
        elif std > 0.08:
            pred.append("running" if std > 0.45 else "walking")
        else:
            pred.append("sitting" if r[ix["tilt_z_deg"]] > 8 else "standing")
    truth = np.where(y == "stairs", "walking", y)
    keep = y != "exercising"
    labels = sorted(set(truth[keep].tolist()))
    return {"macro_f1": float(f1_score(truth[keep], np.array(pred)[keep], average="macro", labels=labels, zero_division=0)),
            "note": "stairs scored as walking, exercising excluded; compare with the model's own macro_f1 above"}


# ------------------------------------------------------------------------------------------------- M2
def train_m2(out: Path = MODELS_DIR, quick: bool = False) -> dict[str, Any]:
    """Fall detection. SisFall was unreachable, so this model is trained on SIMULATED falls and flagged as such."""
    from . import sim_data
    n_pos, n_neg = (60, 180) if quick else (240, 720)
    d = _cache("m2_quick" if quick else "m2", lambda: dict(zip(("X", "y", "g"), sim_data.fall_dataset(n_pos, n_neg), strict=True)))
    X, y, g = d["X"], d["y"].astype(str), d["g"]
    cv = core.cross_val_classify(X, y, g, 5)
    pos = y == "fall"
    from sklearn.metrics import recall_score
    sens = float(recall_score(y == "fall", cv["oof_pred"] == "fall"))
    spec = float(recall_score(y != "fall", cv["oof_pred"] != "fall"))
    data = {"sources": ["edge hardware simulator (randomised fall direction/force + synthetic ADL impacts)"],
            "provenance": "simulated", "n_windows": int(len(y)), "n_falls": int(pos.sum()),
            "notes": "NOT validated on real falls (SisFall/MobiFall not obtainable here). Do not enable on a person "
                     "until retrained and validated on real data: config models.fall.enabled stays false."}
    tgt = {"metric": "sensitivity", "value": 0.95, "got": round(sens, 4), "met": sens >= 0.95,
           "validated_on_real_data": False}
    return core.finish_classifier("m2_fall", X, y, cv, FALL_FEATURES, data, tgt, out,
                                  extra={"sensitivity": sens, "specificity": spec, "enable_on_edge": False})


# ------------------------------------------------------------------------------------------------- M3/M4
def _cinc(quick: bool) -> dict[str, np.ndarray]:
    limit = 500 if quick else None

    def build() -> dict[str, np.ndarray]:
        rr_X, rr_y, rec, q_X, q_y, q_g = [], [], [], [], [], []
        for rid, lab, ecg in datasets.iter_cinc2017(limit=limit):
            seg = ecg[: int(30 * ECG_FS)]
            rr_X.append(rr_features(rr_intervals(r_peaks(seg, ECG_FS), ECG_FS)))
            rr_y.append(lab)
            rec.append(rid)
            for start in (2, 12):
                w = ecg[int(start * ECG_FS):int((start + 5) * ECG_FS)]
                if len(w) >= int(4.5 * ECG_FS):
                    q_X.append(ecg_quality_features(w, ECG_FS))
                    q_y.append("noisy" if lab == "noisy" else "good")
                    q_g.append(rid)
        return {"rr_X": np.stack(rr_X), "rr_y": np.array(rr_y), "rec": np.array(rec), "q_X": np.stack(q_X),
                "q_y": np.array(q_y), "q_g": np.array(q_g)}
    return _cache("cinc_quick" if quick else "cinc", build)


def train_m3(out: Path = MODELS_DIR, quick: bool = False) -> dict[str, Any]:
    """ECG signal quality: real CinC-2017 'noisy' vs readable windows, plus simulated 'unusable' (flat/hum/clipped)."""
    from . import sim_data
    d = _cinc(quick)
    bad = sim_data.unusable_ecg_windows(120 if quick else 600)
    ub = np.stack([ecg_quality_features(w, ECG_FS) for w in bad])
    X = np.concatenate([d["q_X"], ub])
    y = np.concatenate([d["q_y"].astype(str), np.array(["unusable"] * len(ub))])
    g = np.concatenate([d["q_g"].astype(str), np.array([f"sim{i}" for i in range(len(ub))])])
    cv = core.cross_val_classify(X, y, g, 5)
    data = {"sources": ["PhysioNet/CinC 2017 (noisy '~' vs N/A/O recordings, 5 s windows, resampled to 250 Hz)",
                        "simulated unusable windows (flat, 50 Hz hum, clipped, white noise, drift)"],
            "provenance": "mixed", "n_windows": int(len(y)), "class_counts": {c: int((y == c).sum()) for c in sorted(set(y.tolist()))},
            "notes": "CinC 2011 (the dataset named in docs/04) was not reachable; 2017's noisy class is the real-data "
                     "proxy. 'noisy' labels are per recording, so some windows labelled noisy are locally clean."}
    return core.finish_classifier("m3_ecg_quality", X, y, cv, ECG_QUALITY_FEATURES, data,
                                  _f1_target("accuracy", 0.92, cv["accuracy"], True), out)


def train_m4(out: Path = MODELS_DIR, quick: bool = False) -> dict[str, Any]:
    """Rhythm screening on RR features: normal / possible AF / other / noisy (CinC 2017, 30 s)."""
    from sklearn.metrics import f1_score
    d = _cinc(quick)
    X, y, g = d["rr_X"], d["rr_y"].astype(str), d["rec"].astype(str)
    cv = core.cross_val_classify(X, y, g, 5)
    oof = cv["oof_pred"]
    af_f1 = float(f1_score(y == "af", oof == "af"))
    ix = RR_FEATURES.index("cv_rr")
    base_af = float(f1_score(y == "af", X[:, ix] > 0.12))              # the edge's rule: RR variability > 0.12
    data = {"sources": ["PhysioNet/CinC 2017 training set (8,528 single-lead recordings, first 30 s)"],
            "provenance": "real", "n_windows": int(len(y)), "class_counts": {c: int((y == c).sum()) for c in sorted(set(y.tolist()))},
            "notes": "Folds are by recording (CinC does not publish subject ids; most recordings are different people). "
                     "'af' is a screening label 'possible AF', never a diagnosis."}
    tgt = {"metric": "af_f1", "value": 0.80, "got": round(af_f1, 4), "met": af_f1 >= 0.80, "validated_on_real_data": True}
    return core.finish_classifier("m4_rhythm", X, y, cv, RR_FEATURES, data, tgt, out,
                                  extra={"af_f1": af_f1, "rule_baseline_af_f1_rr_cv_gt_0.12": base_af})


# ------------------------------------------------------------------------------------------------- M5
def _m5_rows(p: dict[str, np.ndarray]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One feature row per PAMAP2 window with the exertion history of the previous 5 minutes."""
    feats = np.stack([imu_window_features(w) for w in p["x"]])
    ix = {n: i for i, n in enumerate(IMU_FEATURES)}
    inten = np.clip(feats[:, ix["std_norm"]], 0, 1)
    cad = feats[:, ix["dom_freq"]]
    X, keep = [], []
    for i in range(len(p["y"])):
        same = (p["subject"] == p["subject"][i]) & (p["t"] < p["t"][i]) & (p["t"] >= p["t"][i] - 300)
        load = float(inten[same].mean()) if same.any() else float(inten[i])
        X.append(expected_hr_features(str(p["y"][i]), float(inten[i]), float(cad[i]), load, 12.0))
        keep.append(np.isfinite(p["hr"][i]))
    keep_a = np.array(keep)
    return np.stack(X)[keep_a], p["hr"][keep_a], p["subject"][keep_a], p["t"][keep_a]


def train_m5(out: Path = MODELS_DIR, quick: bool = False) -> dict[str, Any]:
    """Expected HR: gradient-boosted population prior (PAMAP2) + per-user recursive-least-squares correction."""
    from senseband_edge.models.personal import PersonalHr  # the SAME class the band runs
    d = _cache("m5_quick" if quick else "m5", lambda: dict(zip(("X", "hr", "s", "t"), _m5_rows(datasets.load_pamap2(hop=256 if quick else 128)), strict=True)))
    X, hr, s, t = d["X"], d["hr"], d["s"], d["t"]
    pop_err, rls_err = {}, {}
    for subj in sorted(set(s.tolist())):
        te, tr = s == subj, s != subj
        m = core.make_regressor(n_estimators=150).fit(X[tr], hr[tr])
        order = np.argsort(t[te])
        pred = m.predict(X[te][order])
        truth = hr[te][order]
        n_warm = len(truth) // 4                                    # "learning period": first quarter, no scoring
        rls = PersonalHr()
        errs_p, errs_r = [], []
        for i, (pr, tru) in enumerate(zip(pred, truth, strict=True)):
            corrected = rls.correct(float(pr))
            if i >= n_warm:
                errs_p.append(abs(pr - tru))
                errs_r.append(abs(corrected - tru))
            rls.update(float(pr), float(tru))
        pop_err[str(subj)], rls_err[str(subj)] = float(np.mean(errs_p)), float(np.mean(errs_r))
    mae_pop, mae_rls = float(np.mean(list(pop_err.values()))), float(np.mean(list(rls_err.values())))
    final = core.make_regressor(n_estimators=150).fit(X, hr)
    path = core.export_onnx(final, X.shape[1], out / "m5_expected_hr.onnx", regressor=True)
    sess = core.onnx_session(path)
    diff = core.verify_parity(final, sess, X[:300], regressor=True)
    assert diff < 1e-2 and path.stat().st_size < core.MAX_MODEL_BYTES
    rep = {"name": "m5_expected_hr", "algorithm": "lightgbm regressor + RLS personalisation", "feature_version": FEATURE_VERSION,
           "features": EXPECTED_HR_FEATURES, "data": {"sources": ["PAMAP2 (9 subjects, chest HR strap + IMU)"], "provenance": "real",
                                                      "n_windows": int(len(hr)), "notes": "Leave-one-subject-out. Time of day and temperature are constant in PAMAP2 so those inputs are untrained."},
           "cv": {"scheme": "leave-one-subject-out; RLS evaluated prequentially after a 25% learning period",
                  "mae_population_bpm": mae_pop, "mae_personalised_bpm": mae_rls, "per_subject_population": pop_err,
                  "per_subject_personalised": rls_err},
           "target": {"metric": "mae_bpm_after_personalisation", "value": 8.0, "got": round(mae_rls, 2), "met": mae_rls <= 8.0,
                      "validated_on_real_data": True},
           "onnx": {"file": path.name, "bytes": path.stat().st_size, "parity_max_abs_diff": diff},
           "latency_ms_this_machine": core.latency_ms(sess, X.shape[1]), "latency_budget_ms": core.MAX_LATENCY_MS,
           "residual_sd_bpm": float(np.std(final.predict(X) - hr))}
    core.write_report("m5_expected_hr", rep, out)
    return rep


ALL = {"m1": train_m1, "m2": train_m2, "m3": train_m3, "m4": train_m4, "m5": train_m5}


def run(names: list[str], out: Path = MODELS_DIR, quick: bool = False) -> dict[str, dict[str, Any]]:
    res = {}
    for n in names:
        t0 = time.time()
        res[n] = ALL[n](out, quick)
        print(f"{n}: done in {time.time() - t0:.0f}s -> {res[n]['target']}", flush=True)
    return res


In [ ]:
%%writefile senseband_train/report.py
"""Human-readable summary of ml/models/metrics.json -> ml/models/METRICS.md."""
from __future__ import annotations

import json
from pathlib import Path


def write_markdown(models_dir: Path) -> Path:
    m = json.loads((models_dir / "metrics.json").read_text())
    lines = ["# Model metrics", "", "Generated by `python -m senseband_train`. **Read the provenance column**: ",
             "simulated data proves the pipeline, not real-world accuracy.", "",
             "| model | data | provenance | target | result | met | size | ms (this laptop) |", "|---|---|---|---|---|---|---|---|"]
    for name, r in sorted(m.items()):
        t = r["target"]
        lines.append(f"| {name} | {', '.join(r['data']['sources'])[:70]} | {r['data']['provenance']} | "
                     f"{t['metric']} >= {t['value']} | {t['got']} | {'yes' if t['met'] else '**no**'}"
                     f"{'' if t.get('validated_on_real_data') else ' (not real data)'} | {r['onnx']['bytes'] // 1024} KB | "
                     f"{r['latency_ms_this_machine']['median']} |")
    lines += ["", "## Details", ""]
    for name, r in sorted(m.items()):
        lines += [f"### {name}", f"- notes: {r['data'].get('notes', '')}", f"- cv: {r['cv'].get('scheme')}"]
        for k in ("macro_f1", "accuracy", "af_f1", "sensitivity", "specificity", "mae_population_bpm", "mae_personalised_bpm"):
            v = r["cv"].get(k, r.get(k))
            if v is not None:
                lines.append(f"- {k}: {v:.3f}")
        for k, v in r.items():
            if k.startswith("rule_baseline"):
                lines.append(f"- {k}: {v}")
        lines.append("")
    out = models_dir / "METRICS.md"
    out.write_text("\n".join(lines))
    return out


## 3. Download the datasets (safe extraction, ~1 GB)

In [ ]:
import sys, importlib
sys.path.insert(0, '.')
from senseband_train import datasets
for name in ('uci_har', 'cinc2017', 'pamap2'):
    print(name, '->', datasets.fetch(name))

## 4. Train

In [ ]:
import json, warnings
warnings.filterwarnings('ignore')
from senseband_train import train_models as tm
results = {}

In [ ]:
# M3 - ECG quality (fast)
results['m3'] = tm.train_m3()
print(results['m3']['target'])

In [ ]:
# M4 - rhythm / possible AF
results['m4'] = tm.train_m4()
print(results['m4']['target'], 'rule baseline AF-F1:', results['m4']['rule_baseline_af_f1_rr_cv_gt_0.12'])

In [ ]:
# M1 - activity & posture
results['m1'] = tm.train_m1()
print(results['m1']['target'], 'rule baseline:', results['m1']['rule_baseline_on_same_windows'])

In [ ]:
# M5 - expected heart rate (population model + per-person RLS correction)
results['m5'] = tm.train_m5()
print(results['m5']['target'])

## 5. Results

In [ ]:
import pandas as pd
rows = []
for k, r in results.items():
    t = r['target']
    rows.append({'model': r['name'], 'data': r['data']['provenance'], 'metric': t['metric'], 'got': t['got'], 'target': t['value'],
                 'met': t['met'], 'onnx KB': r['onnx']['bytes'] // 1024, 'ms/inference (this machine)': r['latency_ms_this_machine']['median']})
pd.DataFrame(rows)

In [ ]:
print('M1 per-class F1:', {k: round(v, 3) for k, v in results['m1']['cv']['per_class_f1'].items()})
print('M4 per-class F1:', {k: round(v, 3) for k, v in results['m4']['cv']['per_class_f1'].items()})
print('M5 MAE population -> personalised:', round(results['m5']['cv']['mae_population_bpm'], 2), '->', round(results['m5']['cv']['mae_personalised_bpm'], 2))

## 6. Use a trained model (ONNX)

In [ ]:
import numpy as np
from senseband_train import core
from senseband_features import imu_window_features
meta = json.load(open('models/metrics.json'))['m1_activity']
sess = core.onnx_session(core.MODELS_DIR / 'm1_activity.onnx')
rng = np.random.default_rng(0)
walk = np.column_stack([0.3*np.sin(np.linspace(0, 20, 128)), 0.1*np.sin(np.linspace(0, 10, 128)), 1 + 0.25*np.sin(np.linspace(0, 40, 128))])
probs = core.onnx_predict(sess, imu_window_features(walk)[None, :])[0]
print(dict(zip(meta['classes'], probs.round(3))))

## 7. Notes and limitations
* Cross-validation is **subject-wise** (no person in both train and test). CinC does not publish subject ids, so its folds are by recording.
* M1/M5 data come from different sensor placements (waist phone, chest strap); the SenseBand band is worn at the waist, so expect a domain gap.
* M3 mixes real "noisy" labels (CinC 2017) with *simulated* "unusable" windows. M5's time-of-day and temperature inputs are constant in PAMAP2.
* Not met: M1, M4, M5 targets. Next steps are in the SenseBand `ml/README.md`: more waist data for M1, ECG morphology / a small 1-D CNN for M4, a larger HR dataset for M5.
* Models are for research/prototyping. Validate on your own volunteers before any real-world use.